# Evaluation of TREX savings

This notebook evaluates the savings of the TREX representation compared to simple representations. It reads the `merged.csv` file, computes derived columns for savings and entropy reduction, and displays the results.


Meaning of columns:
* `n`: number of nodes
* `m`: number of edges
* `type`: directed or undirected
* `indegree entropy`: H(G), the entropy of the indegree distribution for directed graphs; -1 for undirected graphs
* `indegree entropy greedy`: H(G_greedy), the entropy of the indegree distribution for greedily oriented undirected graphs; -1 for directed graphs
* `array total bits`: total bits for the array representation
* `bitvector total bits`: total bits for the bitvector representation
* `bitvector greedy total bits`: total bits for the bitvector representation of greedily oriented undirected graphs
* `total bits trex`: total bits for the TREX representation
* `trex entropy`: H(G-T), the entropy of the indegree distribution after removing the extracted tree
* `total bits planar`: total bits for the planar representation
* `planar edges`: number of edges in the planar representation

In [ ]:
import math

import pandas as pd

df = pd.read_csv("merged.csv")

# Build a columns for H(G) and the other columns based on the graph type
directed = df["type"].eq("directed")
df["H(G)"] = None
df.loc[directed, "H(G)"] = df.loc[directed, "indegree entropy"]
df.loc[~directed, "H(G)"] = df.loc[~directed, "indegree entropy greedy"]
df["(0) adjacency lists"] = df["array total bits"]
df["(2) Entropy-compressed adjacency string"] = None
df.loc[directed, "(2) Entropy-compressed adjacency string"] = df.loc[directed, "bitvector total bits"]
df.loc[~directed, "(2) Entropy-compressed adjacency string"] = df.loc[~directed, "bitvector greedy total bits"]
df["(3b) TREX"] = df["total bits trex"]
df["H(G-T)"] = df["trex entropy"]

df["n lg n"] = df["n"] * df["n"].map(math.log2)

# Add derived columns for savings and entropy reduction
df["(2)-(3b) tree extraction savings"] = df["(2) Entropy-compressed adjacency string"] - df["(3b) TREX"]
df["(0)-(3b) total savings"] = df["(0) adjacency lists"] - df["(3b) TREX"]
df["H(G)-H(G-T)"] = df["H(G)"] - df["H(G-T)"]

df["(2)-(3b) / n lg n"] = df["(2)-(3b) tree extraction savings"] / df["n lg n"]
df["(H(G)-H(G-T)) / n lg n"] = df["H(G)-H(G-T)"] / df["n lg n"]

df

,Dataset,n,m,type,array total bits,bitvector total bits,total bits trex,trex entropy,bitvector total bits.1,total bits planar,planar edges,tree extraction savings (2)-(3b),total saving (0)-(3b),n lg n,tree extraction savings (2)-(3b) / n lg n
0,Amazon0302.txt,262111,1234877,directed,27732117,2.256989e+07,1.812547e+07,1.641821e+07,2.256989e+07,1.638075e+07,373607,4.444420e+06,9.606647e+06,4.717950e+06,0.942023
1,Cit-HepPh.txt,34546,421578,directed,7401622,5.919122e+06,5.424500e+06,5.148440e+06,5.919122e+06,5.240334e+06,49075,4.946217e+05,1.977122e+06,5.208235e+05,0.949692
2,Slashdot0811.txt,77360,905468,directed,16940156,1.352561e+07,1.231942e+07,1.170611e+07,1.352561e+07,1.222590e+07,85991,1.206188e+06,4.620731e+06,1.256272e+06,0.960133
3,wikilink_graph.2005-03-01_edgelist.txt,244112,3659390,directed,71239484,5.489225e+07,5.061479e+07,4.858880e+07,5.489225e+07,4.956331e+07,317640,4.277464e+06,2.062469e+07,4.368917e+06,0.979067
4,Wiki-Vote.txt,7115,103689,directed,1468912,1.144120e+06,1.079472e+06,1.020727e+06,1.144120e+06,1.066991e+06,8557,6.464788e+04,3.894401e+05,9.104815e+04,0.710041
5,fig2-example.txt,8,14,directed,74,5.322512e+01,4.630321e+01,9.651484e+00,5.322512e+01,4.168728e+01,10,6.921907e+00,2.769679e+01,2.400000e+01,0.288413
6,graph_A_skewed.txt,1000,2997,directed,41970,1.794027e+04,7.757107e+03,2.009406e+03,1.794027e+04,1.189105e+04,1498,1.018317e+04,3.421289e+04,9.965784e+03,1.021813
7,graph_B_uniform.txt,1000,2997,directed,41970,7.987898e+03,7.766514e+03,2.018813e+03,7.987898e+03,7.846935e+03,1001,2.213847e+02,3.420349e+04,9.965784e+03,0.022214
8,G_barabasi_albert.txt,40000,559804,directed,9756864,8.769880e+06,8.274015e+06,7.946154e+06,8.769880e+06,8.169779e+06,47464,4.958657e+05,1.482849e+06,6.115085e+05,0.810889
9,G_bipartite.txt,2000,1000000,directed,11040000,9.986597e+06,9.971229e+06,9.944422e+06,9.986597e+06,9.971412e+06,1999,1.536790e+04,1.068771e+06,2.193157e+04,0.700721
